# RL-дообучение GPT-2 рекомендера на KION

План: данные и разбиение по времени → базовая GPT-2 → reward model → RL (REINFORCE/GRPO + KL) → анализ.
Статус: каркас, секции заполняются по мере работы.

In [ ]:
import os, glob, json, random
from pathlib import Path
import numpy as np

ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = True          # True: малая выборка, быстрый прогон; False: полный прогон на GPU
SEED = 42

WORK = Path("/kaggle/working") if ON_KAGGLE else Path(".")
CACHE, CKPT, RESULTS = WORK/"cache", WORK/"checkpoints", WORK/"results"
for d in (CACHE, CKPT, RESULTS): d.mkdir(exist_ok=True)

def find_data_dir():
    if os.environ.get("DATA_DIR"): return Path(os.environ["DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/interactions.csv", recursive=True)
    if not hits: raise FileNotFoundError("interactions.csv не найден: подключите датасет или задайте DATA_DIR")
    return Path(hits[0]).parent
DATA_DIR = find_data_dir()

random.seed(SEED); np.random.seed(SEED)
print("kaggle:", ON_KAGGLE, "| smoke:", SMOKE, "| data:", DATA_DIR)

## 1. Данные и разбиение по времени

**Протокол разбиения** (один для всех моделей):

- Глобальные даты: train `< 2021-08-02`, val `2021-08-02 … 08-08`, test `2021-08-09 … 08-22` (последние 14 дней).
- Обучение (GPT-2, reward model, RL) использует только train. Val нужна для подбора настроек, test открываем один раз в конце.
- Eval-кейс: пользователь с ≥5 событиями в train и ≥1 событием в периоде. Цель — первое событие в периоде, контекст — все события пользователя до начала периода (для test это train+val). Следующие 5 событий периода сохраняем как «будущее» для проверки 5-шаговой генерации.
- Объекты, которых нет в train, из val/test выкидываем (холодный старт здесь не исследуем), долю считаем и печатаем.
- В KION есть только дата без времени: порядок внутри дня берём из порядка строк файла (стабильная сортировка). Это шум, он описан в ограничениях.
- Пара (пользователь, объект) в данных не повторяется, поэтому уже просмотренные объекты исключаем из рекомендаций при оценке.

In [ ]:
import pandas as pd

TRAIN_END = pd.Timestamp("2021-08-02")   # train: dt < TRAIN_END
VAL_END   = pd.Timestamp("2021-08-09")   # val: [TRAIN_END, VAL_END), test: >= VAL_END
MIN_HIST  = 5                            # минимум событий в train
N_FUTURE  = 5                            # длина «будущего» для многошагового анализа
SMOKE_USERS = 30_000                     # размер выборки пользователей при SMOKE=True
TAG = "smoke" if SMOKE else "full"

In [ ]:
def build_split():
    df = pd.read_csv(DATA_DIR/"interactions.csv", parse_dates=["last_watch_dt"])
    n_raw = len(df)
    df = df.dropna(subset=["watched_pct"]).reset_index(drop=True)
    df["row"] = np.arange(len(df))
    df = df.sort_values(["user_id", "last_watch_dt", "row"], kind="mergesort").reset_index(drop=True)
    df["period"] = np.select([df.last_watch_dt < TRAIN_END, df.last_watch_dt < VAL_END], [0, 1], 2)  # 0 train, 1 val, 2 test

    n_train = df[df.period == 0].groupby("user_id").size()
    keep = n_train[n_train >= MIN_HIST].index
    if SMOKE:
        keep = pd.Index(np.random.RandomState(SEED).choice(keep, SMOKE_USERS, replace=False))
    df = df[df.user_id.isin(keep)].copy()

    # словарь объектов только по train; 0 зарезервирован под PAD
    train_items = np.sort(df.loc[df.period == 0, "item_id"].unique())
    item2idx = {it: i + 1 for i, it in enumerate(train_items)}
    seen = df.item_id.isin(item2idx)
    unseen_share = {p: float((~seen[df.period == p]).mean()) for p in (1, 2)}
    df = df[seen].copy()
    df["item"] = df.item_id.map(item2idx).astype("int32")
    df = df[["user_id", "item", "item_id", "last_watch_dt", "watched_pct", "total_dur", "period"]].reset_index(drop=True)
    return df, item2idx, unseen_share, n_raw

cache_f = CACHE/f"interactions_{TAG}.parquet"
if cache_f.exists():
    df = pd.read_parquet(cache_f)
    item2idx = {it: i + 1 for i, it in enumerate(np.sort(df.loc[df.period == 0, "item_id"].unique()))}
    unseen_share = json.load(open(RESULTS/f"split_stats_{TAG}.json"))["unseen_share"]
else:
    df, item2idx, unseen_share, n_raw = build_split()
    df.to_parquet(cache_f)
n_items = len(item2idx) + 1   # с учётом PAD=0
print(df.shape, "users:", df.user_id.nunique(), "| vocab:", n_items)

Eval-кейсы: цель, контекст (индексы строк в `df`) и «будущее» из 5 событий.

In [ ]:
def build_eval(df, period):
    """Одна строка на пользователя: первое событие периода — цель, следующие N_FUTURE — будущее."""
    cur = df[df.period == period]
    g = cur.groupby("user_id", sort=False)
    first = g.head(1).set_index("user_id")
    fut = g.head(1 + N_FUTURE).groupby("user_id", sort=False).item.apply(lambda s: s.iloc[1:].tolist())
    out = pd.DataFrame({"target": first["item"], "target_pct": first["watched_pct"],
                        "target_dt": first["last_watch_dt"], "future": fut})
    return out

evals = {"val": build_eval(df, 1), "test": build_eval(df, 2)}

def history(user_id, period):
    """Контекст пользователя: все события строго до начала периода."""
    u = df_by_user.get_group(user_id)
    return u[u.period < period]

df_by_user = df.groupby("user_id", sort=False)
for k, e in evals.items(): print(k, "eval users:", len(e))

Проверки на утечки и статистика.

In [ ]:
tr, va, te = (df[df.period == p] for p in (0, 1, 2))
assert tr.last_watch_dt.max() < TRAIN_END
assert va.last_watch_dt.between(TRAIN_END, VAL_END - pd.Timedelta(days=1)).all()
assert te.last_watch_dt.min() >= VAL_END
assert df.groupby("user_id").last_watch_dt.apply(lambda s: s.is_monotonic_increasing).all()   # сортировка по времени
assert set(va.item) <= set(tr.item) and set(te.item) <= set(tr.item)                             # словарь только из train
assert not df.duplicated(["user_id", "item"]).any()                                              # нет повторов, значит цель не из истории
for k, p in (("val", 1), ("test", 2)):                                                           # цель всегда позже истории
    e = evals[k]; hist_last = df[df.period < p].groupby("user_id").last_watch_dt.max()
    assert (e.target_dt.values >= hist_last.reindex(e.index).values).all()

stats = {
    "tag": TAG, "users": int(df.user_id.nunique()), "vocab": int(n_items),
    "events": {"train": len(tr), "val": len(va), "test": len(te)},
    "train_seq_len_median": float(tr.groupby("user_id").size().median()),
    "eval_users": {k: len(e) for k, e in evals.items()},
    "unseen_share": unseen_share,
    "watched_pct_mean": {"train": float(tr.watched_pct.mean()), "val": float(va.watched_pct.mean()), "test": float(te.watched_pct.mean())},
    "share_pct_ge_50": {"train": float((tr.watched_pct >= 50).mean()), "test": float((te.watched_pct >= 50).mean())},
}
json.dump(stats, open(RESULTS/f"split_stats_{TAG}.json", "w"), indent=1)
print(json.dumps(stats, indent=1)); print("утечек не найдено")

## 2. Базовая модель (GPT-2, next item prediction)

**Модель и протокол оценки.**

- GPT-2 (HF `GPT2LMHeadModel`, веса с нуля), токен = индекс объекта, 0 = PAD. Контекст — последние 50 объектов.
- Обучение: train-период, окна по 50 объектов со сдвигом 25, loss = кросс-энтропия next item.
- Оценка: ранжируем весь каталог, **исключая уже просмотренные объекты и PAD**. Единственный релевантный объект — цель кейса, поэтому Recall@K = HitRate@K, а MAP@K = MRR@K.
- Все модели (бейзлайны, база, позже RL) оцениваются одной функцией `evaluate`. Ничья по score считается как худший ранг.
- Доверительные интервалы 95%: бутстреп по пользователям (1000 повторов); для разности моделей — парный бутстреп.
- Разнообразие: покрытие каталога в топ-10, Gini экспозиции объектов, средняя лог-популярность рекомендаций.

In [ ]:
import torch, time
from transformers import GPT2Config, GPT2LMHeadModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN, KS, K_MAX = 50, (5, 10, 20), 20
CFG = (dict(n_layer=2, n_head=4, n_embd=128, epochs=3, bs=64, lr=1e-3) if SMOKE
       else dict(n_layer=4, n_head=4, n_embd=256, epochs=6, bs=256, lr=5e-4))
torch.manual_seed(SEED)
print(DEVICE, CFG)

item_cnt = np.bincount(df.loc[df.period == 0, "item"].values, minlength=n_items)   # популярность в train

def pad(seqs):
    a = np.zeros((len(seqs), max(map(len, seqs))), np.int64)
    for i, s in enumerate(seqs): a[i, :len(s)] = s
    return torch.from_numpy(a)

_hist_cache = {}
def histories(split):
    """Полная история каждого eval-пользователя: все события строго до начала периода."""
    if split in _hist_cache: return _hist_cache[split]
    p = {"val": 1, "test": 2}[split]
    sub = df[df.period < p]; uid, it = sub.user_id.values, sub.item.values
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    pos = pd.Series(np.arange(len(starts)), index=uid[starts]).reindex(evals[split].index).values
    _hist_cache[split] = [it[starts[j]:ends[j]] for j in pos]
    return _hist_cache[split]

@torch.no_grad()
def evaluate(score_fn, split, bs=512):
    """score_fn(ctx[B,L], lens[B]) -> scores[B,V]. Возвращает ранг цели и топ-K_MAX рекомендаций."""
    e, hist = evals[split], histories(split)
    tgt = e.target.values.astype(np.int64)
    ranks, recs = np.empty(len(e), np.int64), np.empty((len(e), K_MAX), np.int64)
    for i in range(0, len(e), bs):
        hb = hist[i:i + bs]
        ctx = pad([h[-MAX_LEN:] for h in hb]); lens = torch.tensor([min(len(h), MAX_LEN) for h in hb])
        sc = score_fn(ctx.to(DEVICE), lens.to(DEVICE)).clone()
        sc[:, 0] = -float("inf")
        rows = torch.from_numpy(np.concatenate([np.full(len(h), j) for j, h in enumerate(hb)]))
        cols = torch.from_numpy(np.concatenate(hb).astype(np.int64))
        sc[rows.to(DEVICE), cols.to(DEVICE)] = -float("inf")           # просмотренное не рекомендуем
        t = torch.from_numpy(tgt[i:i + bs]).to(DEVICE)
        ranks[i:i + bs] = (sc >= sc.gather(1, t[:, None])).sum(1).cpu().numpy()
        recs[i:i + bs] = sc.topk(K_MAX, 1).indices.cpu().numpy()
    return ranks, recs

def user_metrics(ranks):
    m = {}
    for k in KS:
        hit = ranks <= k
        m[f"Recall@{k}"], m[f"NDCG@{k}"], m[f"MAP@{k}"] = hit * 1.0, hit / np.log2(ranks + 1), hit / ranks
    return m

def boot_ci(x, B=1000, seed=0):
    """Среднее и 95% ДИ бутстрепом по пользователям."""
    rng, n, means = np.random.default_rng(seed), len(x), np.empty(B)
    for s in range(0, B, 100):
        means[s:s + 100] = x[rng.integers(0, n, (min(100, B - s), n))].mean(1)
    return [float(x.mean()), *map(float, np.percentile(means, [2.5, 97.5]))]

def diversity(recs, k=10):
    r = recs[:, :k].ravel(); cnt = np.bincount(r, minlength=n_items)[1:]
    p = np.sort(cnt) / cnt.sum(); n = len(p)
    return {"coverage@10": float((cnt > 0).mean()), "gini@10": float((2 * np.arange(1, n + 1) - n - 1) @ p / n),
            "mean_log_pop@10": float(np.log1p(item_cnt[r]).mean())}

def diversity_ci(recs, B=200, seed=0):
    rng, base = np.random.default_rng(seed), diversity(recs)
    boots = [diversity(recs[rng.integers(0, len(recs), len(recs))]) for _ in range(B)]
    q = {k: np.percentile([b[k] for b in boots], [2.5, 97.5]) for k in base}
    out = {k: [v, *map(float, q[k])] for k, v in base.items()}
    out["coverage@10"] = [base["coverage@10"], None, None]   # при пересэмплировании пользователей покрытие смещается вниз: ДИ не считаем
    return out

def report(ranks, recs):
    return {**{k: boot_ci(v) for k, v in user_metrics(ranks).items()}, **diversity_ci(recs)}

def table(res, cols=("Recall@10", "NDCG@10", "MAP@10", "Recall@20", "coverage@10", "gini@10", "mean_log_pop@10")):
    fmt = lambda x: f"{x[0]:.4f}" if x[1] is None else f"{x[0]:.4f} [{x[1]:.4f}, {x[2]:.4f}]"
    return pd.DataFrame({m: {c: fmt(r[c]) for c in cols} for m, r in res.items()}).T

### Бейзлайны: популярное за всё время и за последние 7 дней train

In [ ]:
last7 = df[(df.period == 0) & (df.last_watch_dt >= TRAIN_END - pd.Timedelta(days=7))]
cnt7 = np.bincount(last7.item.values, minlength=n_items)
pop_all = torch.tensor(item_cnt, dtype=torch.float32, device=DEVICE)
pop_7d = torch.tensor(cnt7 + 1e-6 * item_cnt, dtype=torch.float32, device=DEVICE)   # хвост разруливаем общей популярностью

val_res, val_raw = {}, {}
for name, vec in (("Pop (all time)", pop_all), ("Pop (last 7d)", pop_7d)):
    r, rec = evaluate(lambda c, l, v=vec: v[None].expand(len(c), -1), "val")
    val_raw[name] = (r, rec); val_res[name] = report(r, rec)
table(val_res)

### Обучение GPT-2

In [ ]:
def make_windows(stride=MAX_LEN // 2):
    tr = df[df.period == 0]; uid, it = tr.user_id.values, tr.item.values
    starts = np.flatnonzero(np.r_[True, uid[1:] != uid[:-1]]); ends = np.r_[starts[1:], len(uid)]
    out = []
    for s, e in zip(starts, ends):
        seq = it[s:e]
        if len(seq) <= MAX_LEN: out.append(seq); continue
        out += [seq[end - MAX_LEN:end] for end in range(len(seq), MAX_LEN - 1, -stride)]
        if (len(seq) - MAX_LEN) % stride: out.append(seq[:MAX_LEN])
    return out

def new_model():
    cfg = GPT2Config(vocab_size=n_items, n_positions=MAX_LEN, n_embd=CFG["n_embd"], n_layer=CFG["n_layer"],
                     n_head=CFG["n_head"], pad_token_id=0, bos_token_id=0, eos_token_id=0)
    return GPT2LMHeadModel(cfg).to(DEVICE)

def gpt_scores(model):
    def f(ctx, lens):
        h = model.transformer(input_ids=ctx, attention_mask=(ctx != 0).long()).last_hidden_state
        return model.lm_head(h[torch.arange(len(ctx), device=ctx.device), lens - 1]).float()
    return f

def train_base(model, seqs, log_path):
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    steps = CFG["epochs"] * ((len(seqs) + CFG["bs"] - 1) // CFG["bs"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["lr"], total_steps=steps, pct_start=0.1)
    amp = DEVICE == "cuda"; scaler = torch.amp.GradScaler(enabled=amp)
    log, best = [], -1
    for ep in range(CFG["epochs"]):
        model.train(); t0, losses = time.time(), []
        perm = np.random.permutation(len(seqs))
        for i in range(0, len(seqs), CFG["bs"]):
            x = pad([seqs[j] for j in perm[i:i + CFG["bs"]]]).to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=amp):
                loss = model(input_ids=x, attention_mask=(x != 0).long(), labels=x.masked_fill(x == 0, -100)).loss
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        model.eval(); r, _ = evaluate(gpt_scores(model), "val"); rec10 = float((r <= 10).mean())
        log.append(dict(epoch=ep + 1, train_loss=float(np.mean(losses)), val_recall10=rec10, sec=round(time.time() - t0)))
        print(log[-1])
        if rec10 > best: best = rec10; torch.save(model.state_dict(), CKPT/f"base_{TAG}.pt")
    pd.DataFrame(log).to_csv(log_path, index=False)

seqs = make_windows(); print("train windows:", len(seqs))
base = new_model(); print("params:", sum(p.numel() for p in base.parameters()))
if (CKPT/f"base_{TAG}.pt").exists():
    print("загружаю чекпойнт")
else:
    train_base(base, seqs, RESULTS/f"base_train_log_{TAG}.csv")
base.load_state_dict(torch.load(CKPT/f"base_{TAG}.pt", map_location=DEVICE)); _ = base.eval()

### Базовая модель на val против бейзлайнов

In [ ]:
r, rec = evaluate(gpt_scores(base), "val")
val_raw["GPT-2 base"] = (r, rec); val_res["GPT-2 base"] = report(r, rec)
json.dump(val_res, open(RESULTS/f"base_val_{TAG}.json", "w"), indent=1)
display(table(val_res))

# парная разность с лучшим бейзлайном
best_bl = max(("Pop (all time)", "Pop (last 7d)"), key=lambda n: val_res[n]["NDCG@10"][0])
for m in ("Recall@10", "NDCG@10"):
    d = user_metrics(val_raw["GPT-2 base"][0])[m] - user_metrics(val_raw[best_bl][0])[m]
    c = boot_ci(d)
    print(f"{m}: GPT-2 base - {best_bl} = {c[0]:.4f}, 95% ДИ [{c[1]:.4f}, {c[2]:.4f}]")

### Выводы по базовой модели

_Заполнить числами после полного прогона на Kaggle._

## 3. Reward model (watched_pct)

## 4. RL-дообучение (REINFORCE/GRPO + KL)

## 5. Анализ: reward vs релевантность, разнообразие, reward hacking

## 6. Генерация на 5 шагов вперёд

## 7. Варианты функции вознаграждения и компромисс

## 8. Выводы и ограничения